In [1]:
import numpy as np
import pandas as pd
import sklearn
import json
import warnings
import matplotlib.pyplot as plt
import seaborn as sns


In [6]:
import requests

base_url = "https://rest.uniprot.org/uniprotkb/search"
params = {
    "query": "organism_id:9606 AND reviewed:true",
    "fields": "accession,protein_name,length,ft_helix,ft_strand,cc_interaction",
    "format": "tsv",
    "size": 500
}  # the raw content that came back


In [7]:
# 1. Make the API call
response = requests.get(base_url, params=params)

# 2. Check if it worked
if response.status_code != 200:
    print(f"Error: {response.status_code}")
else:
    print("Success!")
    
    # 3. Save to file
    with open("uniprot_proteins.tsv", "w") as f:
        f.write(response.text)
    
    # 4. Read back with pandas and preview
    df = pd.read_csv("uniprot_proteins.tsv", sep="\t")
    print(f"Shape: {df.shape}")
    print(df.head())

Success!
Shape: (500, 6)
        Entry                                      Protein names  Length  \
0  A0A0C5B5G6  Mitochondrial-derived peptide MOTS-c (Mitochon...      16   
1  A0A1B0GTW7  Ciliated left-right organizer metallopeptidase...     788   
2      A0JNW5  Bridge-like lipid transfer protein family memb...    1464   
3      A0JP26               POTE ankyrin domain family member B3     581   
4      A0PK11                                           Clarin-2     232   

  Helix Beta strand                                     Interacts with  
0   NaN         NaN                                                NaN  
1   NaN         NaN                                                NaN  
2   NaN         NaN                                                NaN  
3   NaN         NaN  Q08AG9; O95995; Q9BYQ6; Q9BYR2; P26371; Q9BYQ4...  
4   NaN         NaN  Q8N6S5; O00501; Q9UHP7-3; Q8TBE3; P26715; Q8N3...  


In [8]:
print(df['Interacts with'].isna().sum())
print(df.columns.tolist())

123
['Entry', 'Protein names', 'Length', 'Helix', 'Beta strand', 'Interacts with']


In [9]:
df_clean = df[df['Interacts with'].notna()].reset_index(drop=True)
print(df_clean.shape)
print(df_clean[['Entry', 'Length', 'Interacts with']].head(3))

(377, 6)
    Entry  Length                                     Interacts with
0  A0JP26     581  Q08AG9; O95995; Q9BYQ6; Q9BYR2; P26371; Q9BYQ4...
1  A0PK11     232  Q8N6S5; O00501; Q9UHP7-3; Q8TBE3; P26715; Q8N3...
2  A1A4S6     786                                     Q9UNA1; Q6P5Z2


In [10]:
df_clean['n_interactions'] = df_clean['Interacts with'].str.split(';').apply(len)
print(df_clean[['Entry', 'Length', 'n_interactions']].head(5))

    Entry  Length  n_interactions
0  A0JP26     581               8
1  A0PK11     232              12
2  A1A4S6     786               2
3  A1L190      88              10
4  A1L3X0     281              11


In [11]:
from itertools import combinations

pairs = []
proteins = df_clean['Entry'].tolist()

# FAST - dictionary lookup is O(1)
protein_dict = df_clean.set_index('Entry').to_dict('index')  # built ONCE

for p1, p2 in combinations(proteins, 2):
    row1 = protein_dict[p1]                                   # O(1) lookup
    row2 = protein_dict[p2]                                   # O(1) lookup
    
    length_diff = abs(row1['Length'] - row2['Length'])
    n_int_sum = row1['n_interactions'] + row2['n_interactions']
    
    partners1 = set(row1['Interacts with'].split('; '))
    partners2 = set(row2['Interacts with'].split('; '))
    shared = int(len(partners1 & partners2) > 0)
    
    pairs.append([p1, p2, row1['Length'], row2['Length'],
                  length_diff, n_int_sum, shared])

pairs_df = pd.DataFrame(pairs, columns=[
    'Protein1', 'Protein2', 'Length1', 'Length2',
    'length_diff', 'n_int_sum', 'interacts'
])

print(pairs_df.shape)
print(pairs_df['interacts'].value_counts())
print(pairs_df.head(3))

(70876, 7)
interacts
0    68283
1     2593
Name: count, dtype: int64
  Protein1 Protein2  Length1  Length2  length_diff  n_int_sum  interacts
0   A0JP26   A0PK11      581      232          349         20          0
1   A0JP26   A1A4S6      581      786          205         10          0
2   A0JP26   A1L190      581       88          493         18          0


In [8]:
print(pairs_df['interacts'].value_counts())
print(f"\nPositive rate: {2593/70876*100:.1f}%")
print(f"Total pairs: {len(pairs_df)}")
print(f"\nFeatures available: {pairs_df.columns.tolist()}")

interacts
0    68283
1     2593
Name: count, dtype: int64

Positive rate: 3.7%
Total pairs: 70876

Features available: ['Protein1', 'Protein2', 'Length1', 'Length2', 'length_diff', 'n_int_sum', 'interacts']


In [9]:
# Define X and y
X = pairs_df[[ 'Length1', 'Length2',
    'length_diff', 'n_int_sum'
]]
y = pairs_df['interacts']

# Train/test split - 80% train, 20% test, stratify on y
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                      test_size=0.2, 
                                                      random_state=42, 
                                                      stratify=y)

print(X_train.shape, X_test.shape)
print(y_train.value_counts())

(56700, 4) (14176, 4)
interacts
0    54626
1     2074
Name: count, dtype: int64


In [10]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# Scale features - SVM needs this, RF doesn't


In [13]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) 
X_test_scaled = scaler.transform(X_test)      


In [15]:

# Train SVM
#doing the second time using sm data
svm = SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm.fit(X_train_sm, y_train_sm)

# Evaluate
y_pred_svm = svm.predict(X_test_scaled)
print(classification_report(y_test, y_pred_svm, 
      target_names=['No interaction', 'Interacts']))

                precision    recall  f1-score   support

No interaction       0.99      0.71      0.83     13657
     Interacts       0.09      0.79      0.17       519

      accuracy                           0.71     14176
     macro avg       0.54      0.75      0.50     14176
  weighted avg       0.96      0.71      0.80     14176



In [19]:
from sklearn.ensemble import RandomForestClassifier

# No scaling needed for RF
rf = RandomForestClassifier(n_estimators=100, 
                             class_weight='balanced', 
                             random_state=42)
#used smote as recall was terrible... 27:1 ratio for interacts and not interacts was affecting it

rf.fit(X_train_sm, y_train_sm)

y_pred_rf = rf.predict(X_test)
print(classification_report(y_test, y_pred_rf, 
      target_names=['No interaction', 'Interacts']))

                precision    recall  f1-score   support

No interaction       0.97      0.93      0.95     13657
     Interacts       0.16      0.36      0.22       519

      accuracy                           0.91     14176
     macro avg       0.57      0.64      0.59     14176
  weighted avg       0.94      0.91      0.92     14176



In [18]:
# Install if needed
# pip install imbalanced-learn

from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

print(y_train_sm.value_counts())

interacts
0    54626
1    54626
Name: count, dtype: int64


In [21]:
import os
os.makedirs('results', exist_ok=True)

In [22]:
import joblib
joblib.dump(rf, 'results/rf_model.pkl')
joblib.dump(scaler, 'results/scaler.pkl')
print("Models saved!")

Models saved!
